# Civitas — train Model B (Rewrite + Respond generator)

`bigscience/mt0-base` + LoRA. Model B writes the non-hateful version of a flagged sentence
(Rewrite, used in the live call) and a calm counter-reply (Respond, used in text mode).

## WHAT YOU DO
1. Finish notebook 03a and the review, then upload the **re-packaged** `civitas_gen_data.zip`
   (the one built with `--silver`) to **My Drive → civitas/**.
2. Optional but recommended: if Model A is trained, its `best_model.pt` stays at
   `My Drive/civitas/model_a_v2/best_model.pt`; this notebook then also measures how often
   Model B's outputs are safe.
3. **Runtime → T4 GPU → Run all.** Roughly 1.5–3 h on a T4. Re-run after a disconnect: it
   resumes from `last/` on Drive.
4. Download from Drive → `civitas/model_b_v1/`:
   - `civitas_results_model_b.zip` → `artifacts/`
   - `model_b_adapter.zip` → unzip into `artifacts/model_b/adapter/`

   Then tell Claude "Model B results are in artifacts".
5. Do **not** run "RUN ONCE" until Claude says tuning is finished.

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
GPU = torch.cuda.get_device_name(0)
major, _ = torch.cuda.get_device_capability(0)
BATCH, ACCUM = (32, 1) if "A100" in GPU else (16, 2)
print(GPU, "| precision:", "bf16" if major >= 8 else "fp32 (mT5 overflows in fp16 on T4)",
      f"| batch {BATCH} x accum {ACCUM}")

In [ ]:
# Google Drive: everything that must survive a disconnect lives here
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/civitas"
import os
os.makedirs(PROJECT, exist_ok=True)
print("project folder:", PROJECT, "->", sorted(os.listdir(PROJECT)))

In [ ]:
# Get the code: from GitHub (fill REPO_URL) OR from civitas_code.zip in the Drive folder
REPO_URL = ""  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
import os, subprocess, zipfile, sys
CODE = "/content/civitas"
if REPO_URL:
    if os.path.exists(f"{CODE}/.git"):
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CODE], check=True)
else:
    # made on your PC with `make package-code`
    zpath = f"{PROJECT}/civitas_code.zip"
    assert os.path.exists(zpath), f"Set REPO_URL above, or upload civitas_code.zip to {PROJECT}"
    zipfile.ZipFile(zpath).extractall(CODE)
    inner = [d for d in os.listdir(CODE) if os.path.exists(f"{CODE}/{d}/config.py")]
    if not os.path.exists(f"{CODE}/config.py") and inner:  # zip had a top-level folder
        CODE = f"{CODE}/{inner[0]}"
os.chdir(CODE)
sys.path.insert(0, CODE)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout
      or "code from zip")

In [ ]:
!pip -q install "transformers>=4.40" "peft>=0.10" sentencepiece sacrebleu pyarrow "indic-transliteration>=2.3.45"
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
# Unpack the data package and verify it against its manifest
import hashlib, json, zipfile, os
DATA = "/content/gen_data"
zipfile.ZipFile(f"{PROJECT}/civitas_gen_data.zip").extractall(DATA)
manifest = json.load(open(f"{DATA}/manifest.json"))
for name, digest in manifest["files"].items():
    h = hashlib.sha256(open(f"{DATA}/{name}", "rb").read()).hexdigest()
    assert h == digest, f"{name} is corrupted (sha256 mismatch) - re-upload the zip"
for split, info in manifest["splits"].items():
    print(f"{split:>5}: {info['rows']:>7} rows  languages={info['language_counts']}")
print("built from commit", manifest["git_commit"], "at", manifest["created_at"])

In [ ]:
# Training config. Every key is a train_gen.py flag
RUN_NAME = "model_b_v1"  #@param {type:"string"}
OUT = f"{PROJECT}/{RUN_NAME}"
MODEL_A = f"{PROJECT}/model_a_v2/best_model.pt"
CONFIG = {
    "data_dir": DATA,
    "output_dir": OUT,
    "base": "bigscience/mt0-base",
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
    "epochs": 3,
    "batch_size": BATCH, "grad_accum": ACCUM,
    "lr": 5e-4, "warmup_ratio": 0.05,
    "max_input_length": 128, "max_output_length": 64,
    "evals_per_epoch": 2, "eval_gen_rows": 600, "patience": 3,
    "save_every": 300,
    "sample_power": 0.5,
    "model_a_ckpt": MODEL_A if os.path.exists(MODEL_A) else "",
}
print("safety metric:", "on" if CONFIG["model_a_ckpt"] else "off (no Model A checkpoint on Drive)")
ARGS = " ".join(f"--{k} {v!r}" if isinstance(v, str) else f"--{k} {v}" for k, v in CONFIG.items())
print(ARGS)

In [ ]:
# Train. Safe to re-run after a disconnect (--resume)
!cd {CODE} && python -m model_b_generation.train_gen {ARGS} --resume

## Results (saved to files)
Writes `OUT/civitas_results_model_b.zip` (metrics, history, samples) and `OUT/model_b_adapter.zip`.

In [ ]:
import json, shutil
m = json.load(open(f"{OUT}/results/metrics.json"))
print({k: m.get(k) for k in ("chrf", "copy_rate", "safety_rate", "val_loss", "selection_score", "global_step")})
for g, v in m.get("by_group", {}).items():
    print(f"  {g:<22} chrF {v['chrf']:>5}  copy {v['copy_rate']:.2f}  safety {v['safety_rate']}")
!cd {CODE} && python -m results_io --run_dir {OUT} --zip {OUT}/civitas_results_model_b.zip
shutil.make_archive(f"{OUT}/model_b_adapter", "zip", f"{OUT}/best")
print("download:", f"{OUT}/civitas_results_model_b.zip", "and", f"{OUT}/model_b_adapter.zip")

---
# RUN ONCE — final report
Only after Claude confirms tuning is finished. Scores the **test** split once.

In [ ]:
CONFIRM = False  #@param {type:"boolean"}
assert CONFIRM, "Tick CONFIRM only when tuning is finished."
!cd {CODE} && python -m model_b_generation.evaluate_gen --data_dir {DATA} --run_dir {OUT} --model_a_ckpt "{CONFIG['model_a_ckpt']}"
!cd {CODE} && python -m results_io --run_dir {OUT} --zip {OUT}/civitas_results_model_b.zip
print("download again:", f"{OUT}/civitas_results_model_b.zip")